# `data/final` CSV 数据质量审计

## tl;dr

- 本次只检查 33 个 CSV，不检查任何 MOL2 文件。
- 两个确定的上游解析缺陷已经进入 final：离子角色误判，以及合法数值 `0` 被转成缺失。
- final 表没有整行精确重复或非有限数值，但若不先修复上述问题，不建议直接作为最终训练集发布。

## Context & Methods

审计覆盖完整性、唯一性、数值有效性、离子角色、预期粒度冲突、跨表覆盖以及处理逻辑。关键检查由同目录下的 `audit.py` 实现。

### Key Assumptions

- `cation` 应具有正净形式电荷，`anion` 应具有负净形式电荷，且二者不应相同。
- 单属性表的候选粒度为所有系统标识列与实验条件列；同一粒度多行意味着同一输入对应多个标签。
- 数值零是合法数据，不能被通用数值解析器当作缺失。
- 没有时间字段，因此无法判断数据新鲜度或历史漂移。

## Data

加载 `data/final` 下所有 CSV，并执行可复现审计。

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / 'data' / 'final').exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.final_data_quality.audit import run_audit

audit = run_audit(PROJECT_ROOT)
pd.Series(audit['totals'], name='value').to_frame()

,value
csv_files,33
rows,1453746
exact_duplicate_rows,0
nonfinite_numeric_cells,0


## Results

### 离子角色和零值丢失

In [2]:
display(audit['zero_loss'])
display(audit['ion_roles'])

,field,raw_zero_values,final_missing_or_dropped,final_context
0,charge,20206,20208,neutral rows absent from simulation/charge.csv
1,ESP_pos_frac,6752,6749,zero fractions became null after rejected/dupl...
2,Dipole,48,46,zero dipoles became null after rejected/duplic...
3,transfer_organic_kcal/mol,32,32,zero labels were rejected as missing


,file,rows,cation_equals_anion,invalid_ion_role_rows,invalid_ion_role_rate
0,experiment/density.csv,97952,284,287,0.002930
1,experiment/electrical_conductivity.csv,11109,3,3,0.000270
2,experiment/equilibrium_pressure.csv,2029,27,27,0.013307
3,experiment/glass_transition_temperature.csv,793,18,18,0.022699
4,experiment/heat_capacity.csv,23584,83,83,0.003519
5,experiment/melting_point.csv,4709,246,248,0.052665
6,experiment/pec50.csv,334,0,3,0.008982
7,experiment/refractive_index.csv,10761,47,47,0.004368
8,experiment/static_relative_permittivity.csv,65,1,1,0.015385
9,experiment/surface_tension.csv,11666,20,21,0.001800


### 粒度冲突、QM 表与映射表

In [3]:
display(audit['grain_conflicts'].head(10))
display(pd.Series(audit['qm'], name='value').to_frame())
display(pd.Series(audit['mapping'], name='value').to_frame())

,file,label,rows,conflict_groups,affected_rows,affected_rate,median_range,p95_range,max_range,max_group_size
0,experiment/solvation.csv,solvation_kcal/mol,44299,20517,41035,0.926319,0.000248,0.000474,0.004156,3
1,experiment/melting_point.csv,melting_point_K,4709,737,1913,0.406243,0.200000,45.240000,265.100000,22
2,experiment/viscosity.csv,viscosity_mPa*s_log10,44529,4027,12289,0.275977,0.027641,0.313168,1.486753,20
3,experiment/density.csv,density_g/cm^3,97952,7761,24675,0.251909,0.001380,0.019700,0.356170,33
4,experiment/self_diffusion_coefficient.csv,self_diffusion_coefficient_10^-9*m^2/s_log10,382,42,87,0.227749,0.104998,0.280077,0.574031,4
5,experiment/surface_tension.csv,surface_tension_mN/m,11666,960,2311,0.198097,0.510000,6.551000,65.480000,9
6,experiment/refractive_index.csv,refractive_index_unitless,10761,793,2131,0.198030,0.001200,0.011000,0.044000,11
7,experiment/static_relative_permittivity.csv,static_relative_permittivity_unitless,65,5,12,0.184615,2.400000,5.820000,6.600000,3
8,experiment/speed_of_sound.csv,speed_of_sound_m/s,6819,412,1088,0.159554,1.400000,14.245000,37.300000,10
9,experiment/heat_capacity.csv,heat_capacity_J/mol/K,23584,1354,3469,0.147091,9.000000,57.350000,228.000000,11


,value
rows,28217.000000
unique_smiles,27279.000000
duplicate_smiles_groups,737.000000
affected_duplicate_rows,1675.000000
max_rows_per_smiles,24.000000
median_gap_spread_eV,0.097145
max_gap_spread_eV,3.418840
negative_gap_rows,2.000000
missing_label_cells,6801.000000


,value
rows,28214
unique_mol_id,28214
invalid_smiles_rows,2
noncanonical_smiles_rows,203
noncanonical_unique_smiles,202


### 实验压力缺失

In [4]:
audit['pressure_missingness']

,file,rows,missing_pressure_rows,missing_pressure_rate
0,experiment/viscosity.csv,44529,19866,0.446136
1,experiment/heat_capacity.csv,23584,10349,0.438814
2,experiment/refractive_index.csv,10761,2940,0.273209
3,experiment/electrical_conductivity.csv,11109,2255,0.202989
4,experiment/density.csv,97952,19145,0.195453
5,experiment/thermal_conductivity.csv,1597,13,0.008140


## Takeaways

1. 先修正 `to_float(0)` 与离子拆分逻辑，再全量重建 final。
2. 对 `solvation.csv` 使用属性感知的近似值合并规则；其他重复测量需保留复现实验标识或明确聚合策略。
3. 给 QM 表补充 `mol_id`/构象/计算批次键，并将零值恢复为零。
4. 为 final 增加自动化门禁：离子净电荷、零值保真、候选键、SMILES 有效性和跨表覆盖率。